#1 Txt encoder 0.01

In [ ]:
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt

# ==========================================
# 1. Self-Attention Text Encoder (Y-Encoder)
# ==========================================
# class SelfAttentionYEncoder:
#     def __init__(self, vocab_size, max_seq_len, embed_dim, target_dim):
#         np.random.seed(42)
#         self.embed_dim = embed_dim
#         self.d_k = embed_dim
        
#         # High variance (* 1.0) to spread embeddings apart and prevent mode collapse
#         self.token_embeddings = np.random.randn(vocab_size, embed_dim) * 1.0
#         self.position_embeddings = np.random.randn(max_seq_len, embed_dim) * 1.0
#         self.W_Q = np.random.randn(embed_dim, self.d_k) * 1.0
#         self.W_K = np.random.randn(embed_dim, self.d_k) * 1.0
#         self.W_V = np.random.randn(embed_dim, self.d_k) * 1.0
#         self.W_Output = np.random.randn(embed_dim, target_dim) * 1.0

#     def softmax(self, z):
#         exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
#         return exp_z / np.sum(exp_z, axis=1, keepdims=True)

#     def encode(self, token_indices):
#         """Outputs the target embedding S_Y as a column vector (target_dim, 1)"""
#         seq_length = len(token_indices)
#         X = self.token_embeddings[token_indices] + self.position_embeddings[:seq_length]
        
#         Q = np.dot(X, self.W_Q)
#         K = np.dot(X, self.W_K)
#         V = np.dot(X, self.W_V)
        
#         scores = np.dot(Q, K.T) / np.sqrt(self.d_k)
#         attention_weights = self.softmax(scores)
#         contextual_embeddings = np.dot(attention_weights, V)
        
#         sentence_representation = np.mean(contextual_embeddings, axis=0)
#         S_Y = np.dot(sentence_representation, self.W_Output)
        
#         return S_Y.reshape(-1, 1)
vocab = [
"a",
"person",
"walking",
"boxing",
"clapping"
]

class SimpleTextEncoder:
    def __init__(self, vocab, embed_dim=64): # FIXED: Double underscores added
        np.random.seed(42)

        self.embed_dim = embed_dim
        self.vocab = vocab

        # Map words → index
        self.word_to_idx = {w:i for i,w in enumerate(vocab)}

        # Trainable word embeddings
        # FIXED: Increased variance to 1.0 to prevent shared words from diluting the meaning
        self.embeddings = np.random.randn(len(vocab), embed_dim) * 1.0 

    def encode(self, words):
        idxs = [self.word_to_idx[w] for w in words]
        vecs = self.embeddings[idxs]

        # Sentence embedding = mean of word vectors
        sentence = np.mean(vecs, axis=0)

        return sentence.reshape(-1,1)

# ==========================================
# 2. InfoNCE Contrastive Loss
# ==========================================
class InfoNCELoss:
    def __init__(self, temperature=0.1):
        self.tau = temperature

    def forward(self, S_y_hat, S_y):
        self.batch_size = S_y_hat.shape[1]
        
        # L2 Normalize predictions (P) and targets (T)
        self.norm_P = np.linalg.norm(S_y_hat, axis=0, keepdims=True) + 1e-8
        self.norm_T = np.linalg.norm(S_y, axis=0, keepdims=True) + 1e-8
        
        self.P = S_y_hat / self.norm_P
        self.T = S_y / self.norm_T
        
        # Cosine Similarity Matrix scaled by temperature
        self.logits = np.dot(self.P.T, self.T) / self.tau
        
        # Softmax over the columns
        exp_logits = np.exp(self.logits - np.max(self.logits, axis=1, keepdims=True))
        self.softmax = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        
        # Calculate cross-entropy on the diagonal (correct matches)
        target_probs = np.diag(self.softmax)
        loss = -np.mean(np.log(target_probs + 1e-8))
        
        return loss

    def backward(self, S_y_hat):
        # Gradient of cross-entropy wrt logits
        d_logits = self.softmax.copy()
        np.fill_diagonal(d_logits, np.diag(d_logits) - 1)
        d_logits /= self.batch_size
        
        # Gradient wrt normalized predictions (P)
        d_P = np.dot(self.T, d_logits.T) / self.tau
        
        # Backpropagate through L2 normalization
        dot_dP_P = np.sum(d_P * self.P, axis=0, keepdims=True)
        d_S_y_hat = (d_P - self.P * dot_dP_P) / self.norm_P
        
        return d_S_y_hat

# ==========================================
# 3. The VL-JEPA Model
# ==========================================
class FinalNumPyJEPA:
    def __init__(self, input_dim, embed_dim, text_encoder, lr=0.01):
        self.lr = lr
        self.text_encoder = text_encoder
        self.criterion = InfoNCELoss(temperature=0.1)
        
        # Vision Encoder Weights
        self.W_v = np.random.randn(embed_dim, input_dim) * np.sqrt(2. / input_dim)
        self.b_v = np.zeros((embed_dim, 1))
        
        # Predictor Weights
        self.W_p = np.random.randn(embed_dim, embed_dim) * np.sqrt(2. / embed_dim)
        self.b_p = np.zeros((embed_dim, 1))

    def relu(self, Z): return np.maximum(0, Z)
    def relu_deriv(self, Z): return Z > 0

    def forward(self, X_v_batch, text_tokens_batch):
        # 1. Vision Encoding (Processes whole batch)
        Z_v = np.dot(self.W_v, X_v_batch) + self.b_v
        self.S_v = self.relu(Z_v)
        
        # 2. Text Encoding (Process each sequence, then stack horizontally)
        S_y_list = [self.text_encoder.encode(tokens) for tokens in text_tokens_batch]
        self.S_y = np.hstack(S_y_list) # Shape: (embed_dim, batch_size)
        
        # 3. Prediction
        self.S_y_hat = np.dot(self.W_p, self.S_v) + self.b_p
        
        # 4. InfoNCE Loss calculation
        loss = self.criterion.forward(self.S_y_hat, self.S_y)
        return loss, Z_v

    def backward(self, X_v_batch, Z_v):
        # Gradients from InfoNCE
        d_S_y_hat = self.criterion.backward(self.S_y_hat)
        
        # Backpropagate Predictor
        d_W_p = np.dot(d_S_y_hat, self.S_v.T)
        d_b_p = np.sum(d_S_y_hat, axis=1, keepdims=True)
        
        # Backpropagate Vision Encoder
        d_S_v = np.dot(self.W_p.T, d_S_y_hat)
        d_Z_v = d_S_v * self.relu_deriv(Z_v)
        d_W_v = np.dot(d_Z_v, X_v_batch.T)
        d_b_v = np.sum(d_Z_v, axis=1, keepdims=True)
        
        # Update Weights
        self.W_p -= self.lr * d_W_p
        self.b_p -= self.lr * d_b_p
        self.W_v -= self.lr * d_W_v
        self.b_v -= self.lr * d_b_v

# ==========================================
# 4. Evaluation / Prediction
# ==========================================
def predict(model, X_v, num_classes, text_mapping):
    # Ensure X_v is a column vector
    X_v = X_v.reshape(-1, 1)
    
    # FIXED: Added normalization to match the training loop!
    X_v = X_v / (np.linalg.norm(X_v, axis=0, keepdims=True) + 1e-8)
    
    # 1. Vision encoder & Predictor
    Z_v = np.dot(model.W_v, X_v) + model.b_v
    S_v = model.relu(Z_v)
    S_y_hat = np.dot(model.W_p, S_v) + model.b_p

    distances = []
    
    # 2. Compare predicted embedding to all possible text targets
    for c in range(num_classes):
        tokens = text_mapping[c]
        S_y = model.text_encoder.encode(tokens)
        
        # Calculate distance using Cosine Similarity
        sim = np.dot(S_y_hat.T, S_y) / (
            np.linalg.norm(S_y_hat) * np.linalg.norm(S_y) + 1e-8
        )
        dist = -sim.item()  # Convert to scalar and negate for distance
        distances.append(dist)

    # Return the index of the closest text embedding
    return np.argmin(distances)

# ==========================================
# 5. Training Loop
# ==========================================
def run_training_and_plot(csv_path, epochs=400, batch_size=4, num_classes=3):
    print(f"Loading Motion Features from {csv_path}...")
    try:
        df = pd.read_csv(csv_path)
        feature_col = 'features' if 'features' in df.columns else 'hsv_features'
        df[feature_col] = df[feature_col].apply(ast.literal_eval)
    except Exception as e:
        print(f"Error: {e}")
        return

    # Distinct tokens to prevent mode collapse
    text_mapping = {
        0: ["a","person","walking"],
        1: ["a","person","boxing"],
        2: ["a","person","clapping"]   # "a person handclapping"
    }

    # Initialize Text Encoder and Model
    text_encoder = SimpleTextEncoder(vocab=vocab, embed_dim=64)
    model = FinalNumPyJEPA(input_dim=256, embed_dim=64, text_encoder=text_encoder, lr=0.05)
    loss_history = []

    num_samples = len(df)
    print(f"Starting InfoNCE Batched Training on {num_samples} samples...")
    
    for epoch in range(epochs):
        df_shuffled = df.sample(frac=1).reset_index(drop=True)
        epoch_loss = 0
        num_batches = 0

        # Process in batches
        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_df = df_shuffled.iloc[start_idx:end_idx]
            
            # InfoNCE needs multiple samples to push away from; skip batch if size is 1
            if len(batch_df) < 2: 
                continue

            # Gather batch data
            X_v_batch = []
                       
            text_tokens_batch = []
            for _, row in batch_df.iterrows():
                X_v_batch.append(row[feature_col])
                text_tokens_batch.append(text_mapping[int(row['target_class'])])
                
            # Transpose to make X_v_batch shape (256, batch_size)
            X_v_batch = np.array(X_v_batch).T

            #Normalize batch features
            X_v_batch = X_v_batch / (np.linalg.norm(X_v_batch, axis=0, keepdims=True) + 1e-8)

            # Forward & Backward Pass
            loss, Z_v = model.forward(X_v_batch, text_tokens_batch)
            model.backward(X_v_batch, Z_v)
            
            epoch_loss += loss
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        loss_history.append(avg_loss)
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | InfoNCE Loss: {avg_loss:.6f}")

    # Evaluation Phase
    correct = 0
    for _, row in df.iterrows():
        X_v = np.array(row[feature_col])
        pred = predict(model, X_v, num_classes, text_mapping)
        if pred == int(row['target_class']):
            correct += 1
    
    print(f"\n--- Training Results ---")
    print(f"Final Accuracy: {(correct/len(df))*100:.2f}%")

    # Plotting
    plt.figure(figsize=(8, 4))
    plt.plot(loss_history, color='blue')
    plt.title('VL-JEPA: InfoNCE Contrastive Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.show()
    
    return model

# ==========================================
# 6. Execution
# ==========================================
if __name__ == "__main__":
    # Ensure this points to your correct local file path
    csv_file = r"C:\Users\subhe\OneDrive\Desktop\VL-JEPA\codes\VL-JEPA-base-CCTV\kth_spatial_motion3.csv"
    trained_model = run_training_and_plot(csv_file, epochs=400, batch_size=4)

#2 Text encoder with InfoNCE loss.

In [ ]:
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt

# ==========================================
# 1. Simple Text Encoder (Y-Encoder)
# ==========================================
vocab = [
    "a",
    "person",
    "walking",
    "boxing",
    "clapping"
]

class SimpleTextEncoder:
    def __init__(self, vocab, embed_dim=64): 
        np.random.seed(42)
        self.embed_dim = embed_dim
        self.vocab = vocab
        self.word_to_idx = {w:i for i,w in enumerate(vocab)}
        
        # High variance (1.0) to prevent shared words from diluting the meaning
        self.embeddings = np.random.randn(len(vocab), embed_dim) * 1.0 

    def encode(self, words):
        idxs = [self.word_to_idx[w] for w in words]
        vecs = self.embeddings[idxs]
        sentence = np.mean(vecs, axis=0)
        return sentence.reshape(-1,1)

# ==========================================
# 2. InfoNCE Contrastive Loss
# ==========================================
class InfoNCELoss:
    def __init__(self, temperature=0.1):
        self.tau = temperature

    def forward(self, S_y_hat, S_y):
        self.batch_size = S_y_hat.shape[1]
        
        self.norm_P = np.linalg.norm(S_y_hat, axis=0, keepdims=True) + 1e-8
        self.norm_T = np.linalg.norm(S_y, axis=0, keepdims=True) + 1e-8
        
        self.P = S_y_hat / self.norm_P
        self.T = S_y / self.norm_T
        
        self.logits = np.dot(self.P.T, self.T) / self.tau
        
        exp_logits = np.exp(self.logits - np.max(self.logits, axis=1, keepdims=True))
        self.softmax = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        
        target_probs = np.diag(self.softmax)
        loss = -np.mean(np.log(target_probs + 1e-8))
        return loss

    def backward(self, S_y_hat):
        d_logits = self.softmax.copy()
        np.fill_diagonal(d_logits, np.diag(d_logits) - 1)
        d_logits /= self.batch_size
        
        d_P = np.dot(self.T, d_logits.T) / self.tau
        
        dot_dP_P = np.sum(d_P * self.P, axis=0, keepdims=True)
        d_S_y_hat = (d_P - self.P * dot_dP_P) / self.norm_P
        return d_S_y_hat

# ==========================================
# 3. The VL-JEPA Model (UNCOMMENTED!)
# ==========================================
class FinalNumPyJEPA:
    def __init__(self, input_dim, embed_dim, text_encoder, lr=0.01):
        self.lr = lr
        self.text_encoder = text_encoder
        self.criterion = InfoNCELoss(temperature=0.1)
        
        # Vision Encoder Weights
        self.W_v = np.random.randn(embed_dim, input_dim) * np.sqrt(2. / input_dim)
        self.b_v = np.zeros((embed_dim, 1))
        
        # Predictor Weights
        self.W_p = np.random.randn(embed_dim, embed_dim) * np.sqrt(2. / embed_dim)
        self.b_p = np.zeros((embed_dim, 1))

    def relu(self, Z): return np.maximum(0, Z)
    def relu_deriv(self, Z): return Z > 0

    def forward(self, X_v_batch, text_tokens_batch):
        # 1. Vision Encoding 
        Z_v = np.dot(self.W_v, X_v_batch) + self.b_v
        self.S_v = self.relu(Z_v)
        
        # 2. Text Encoding
        S_y_list = [self.text_encoder.encode(tokens) for tokens in text_tokens_batch]
        self.S_y = np.hstack(S_y_list) 
        
        # 3. Prediction
        self.S_y_hat = np.dot(self.W_p, self.S_v) + self.b_p
        
        # 4. InfoNCE Loss
        loss = self.criterion.forward(self.S_y_hat, self.S_y)
        return loss, Z_v

    def backward(self, X_v_batch, Z_v):
        d_S_y_hat = self.criterion.backward(self.S_y_hat)
        
        d_W_p = np.dot(d_S_y_hat, self.S_v.T)
        d_b_p = np.sum(d_S_y_hat, axis=1, keepdims=True)
        
        d_S_v = np.dot(self.W_p.T, d_S_y_hat)
        d_Z_v = d_S_v * self.relu_deriv(Z_v)
        d_W_v = np.dot(d_Z_v, X_v_batch.T)
        d_b_v = np.sum(d_Z_v, axis=1, keepdims=True)
        
        self.W_p -= self.lr * d_W_p
        self.b_p -= self.lr * d_b_p
        self.W_v -= self.lr * d_W_v
        self.b_v -= self.lr * d_b_v

# ==========================================
# 4. Evaluation / Prediction
# ==========================================
def predict(model, X_v, num_classes, text_mapping):
    X_v = X_v.reshape(-1, 1)
    
    # Normalization matches the training loop!
    X_v = X_v / (np.linalg.norm(X_v, axis=0, keepdims=True) + 1e-8)
    
    Z_v = np.dot(model.W_v, X_v) + model.b_v
    S_v = model.relu(Z_v)
    S_y_hat = np.dot(model.W_p, S_v) + model.b_p

    distances = []
    
    for c in range(num_classes):
        tokens = text_mapping[c]
        S_y = model.text_encoder.encode(tokens)
        
        # Cosine Similarity
        sim = np.dot(S_y_hat.T, S_y) / (np.linalg.norm(S_y_hat) * np.linalg.norm(S_y) + 1e-8)
        dist = -sim.item()  
        distances.append(dist)

    return np.argmin(distances)

# ==========================================
# 5. Training Loop
# ==========================================
def run_training_and_plot(csv_path, epochs=400, batch_size=4, num_classes=3):
    print(f"Loading Motion Features from {csv_path}...")
    try:
        df = pd.read_csv(csv_path)
        feature_col = 'features' if 'features' in df.columns else 'hsv_features'
        df[feature_col] = df[feature_col].apply(ast.literal_eval)
    except Exception as e:
        print(f"Error: {e}")
        return

    text_mapping = {
        0: ["a","person","walking"],
        1: ["a","person","boxing"],
        2: ["a","person","clapping"]   
    }

    text_encoder = SimpleTextEncoder(vocab=vocab, embed_dim=64)
    # Using the correct FinalNumPyJEPA model!
    model = FinalNumPyJEPA(input_dim=256, embed_dim=64, text_encoder=text_encoder, lr=0.05)
    loss_history = []

    num_samples = len(df)
    print(f"Starting InfoNCE Batched Training on {num_samples} samples...")
    
    for epoch in range(epochs):
        df_shuffled = df.sample(frac=1).reset_index(drop=True)
        epoch_loss = 0
        num_batches = 0

        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_df = df_shuffled.iloc[start_idx:end_idx]
            
            if len(batch_df) < 2: 
                continue

            X_v_batch = []
            text_tokens_batch = []
            for _, row in batch_df.iterrows():
                X_v_batch.append(row[feature_col])
                text_tokens_batch.append(text_mapping[int(row['target_class'])])
                
            X_v_batch = np.array(X_v_batch).T
            X_v_batch = X_v_batch / (np.linalg.norm(X_v_batch, axis=0, keepdims=True) + 1e-8)

            loss, Z_v = model.forward(X_v_batch, text_tokens_batch)
            model.backward(X_v_batch, Z_v)
            
            epoch_loss += loss
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        loss_history.append(avg_loss)
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | InfoNCE Loss: {avg_loss:.6f}")

    # Evaluation Phase
    correct = 0
    for _, row in df.iterrows():
        X_v = np.array(row[feature_col])
        pred = predict(model, X_v, num_classes, text_mapping)
        if pred == int(row['target_class']):
            correct += 1
    
    print(f"\n--- Training Results ---")
    print(f"Final Accuracy: {(correct/len(df))*100:.2f}%")

    plt.figure(figsize=(8, 4))
    plt.plot(loss_history, color='blue')
    plt.title('VL-JEPA: InfoNCE Contrastive Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.show()
    
    return model

# ==========================================
# 6. Execution
# ==========================================
if __name__ == "__main__":
    csv_file = r"C:\Users\subhe\OneDrive\Desktop\VL-JEPA\codes\VL-JEPA-base-CCTV\kth_spatial_motion3.csv"
    trained_model = run_training_and_plot(csv_file, epochs=400, batch_size=4)

#3 vision + text encoder

In [ ]:
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt

# ==========================================
# 1. VISION TRANSFORMER (X-Encoder)
# ==========================================
class PatchEmbedding:
    def __init__(self, img_size=224, patch_size=16, in_ch=3, embed_dim=256):
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.num_patches = (img_size // patch_size) ** 2
        patch_dim = patch_size * patch_size * in_ch
        self.W = np.random.randn(patch_dim, embed_dim) / np.sqrt(patch_dim)

    def forward(self, x):
        B, H, W, C = x.shape
        p = self.patch_size
        patches = []
        for i in range(0, H, p):
            for j in range(0, W, p):
                patch = x[:, i:i+p, j:j+p, :].reshape(B, -1)
                patches.append(patch)
        patches = np.stack(patches, axis=1)   # (B, N, patch_dim)
        return patches @ self.W               # (B, N, embed_dim)

class LayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.gamma = np.ones((1, 1, dim))
        self.beta = np.zeros((1, 1, dim))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

class MLP:
    def __init__(self, dim, hidden_dim):
        self.W1 = np.random.randn(dim, hidden_dim) / np.sqrt(dim)
        self.W2 = np.random.randn(hidden_dim, dim) / np.sqrt(hidden_dim)

    def gelu(self, x):
        return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * np.power(x, 3))))

    def forward(self, x):
        return self.gelu(x @ self.W1) @ self.W2

class MultiHeadSelfAttention:
    def __init__(self, dim, heads=8):
        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.Wq = np.random.randn(dim, dim) / np.sqrt(dim)
        self.Wk = np.random.randn(dim, dim) / np.sqrt(dim)
        self.Wv = np.random.randn(dim, dim) / np.sqrt(dim)
        self.Wo = np.random.randn(dim, dim) / np.sqrt(dim)

    def forward(self, x):
        B, N, D = x.shape
        Q = x @ self.Wq
        K = x @ self.Wk
        V = x @ self.Wv

        Q = Q.reshape(B, N, self.heads, self.head_dim).transpose(0,2,1,3)
        K = K.reshape(B, N, self.heads, self.head_dim).transpose(0,2,1,3)
        V = V.reshape(B, N, self.heads, self.head_dim).transpose(0,2,1,3)

        scores = Q @ K.transpose(0,1,3,2) / np.sqrt(self.head_dim)
        
        # Stability adjustment for Softmax
        scores = scores - np.max(scores, axis=-1, keepdims=True)
        attn = np.exp(scores)
        attn = attn / attn.sum(axis=-1, keepdims=True)

        out = attn @ V
        out = out.transpose(0,2,1,3).reshape(B, N, D)
        return out @ self.Wo

class TransformerBlock:
    def __init__(self, dim, heads):
        self.norm1 = LayerNorm(dim)
        self.norm2 = LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, heads)
        self.mlp = MLP(dim, dim*4)

    def forward(self, x):
        x = x + self.attn.forward(self.norm1.forward(x))
        x = x + self.mlp.forward(self.norm2.forward(x))
        return x

class NumPyViT:
    def __init__(self, img_size=224, patch_size=16, embed_dim=256, depth=4, heads=4):
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        self.pos_embed = np.random.randn(1, self.patch_embed.num_patches, embed_dim)
        self.blocks = [TransformerBlock(embed_dim, heads) for _ in range(depth)]
        self.norm = LayerNorm(embed_dim)

    def forward(self, x):
        x = self.patch_embed.forward(x)
        x = x + self.pos_embed
        for blk in self.blocks:
            x = blk.forward(x)
        return self.norm.forward(x)

# ==========================================
# 2. TEXT ENCODER & INFONCE LOSS
# ==========================================
vocab = ["a", "person", "walking", "boxing", "clapping"]

class SimpleTextEncoder:
    def __init__(self, vocab, embed_dim=64): 
        np.random.seed(42)
        self.embed_dim = embed_dim
        self.word_to_idx = {w:i for i,w in enumerate(vocab)}
        self.embeddings = np.random.randn(len(vocab), embed_dim) * 1.0 

    def encode(self, words):
        idxs = [self.word_to_idx[w] for w in words]
        sentence = np.mean(self.embeddings[idxs], axis=0)
        return sentence.reshape(-1, 1)

class InfoNCELoss:
    def __init__(self, temperature=0.1):
        self.tau = temperature

    def forward(self, S_y_hat, S_y):
        self.batch_size = S_y_hat.shape[1]
        self.norm_P = np.linalg.norm(S_y_hat, axis=0, keepdims=True) + 1e-8
        self.norm_T = np.linalg.norm(S_y, axis=0, keepdims=True) + 1e-8
        
        self.P = S_y_hat / self.norm_P
        self.T = S_y / self.norm_T
        
        self.logits = np.dot(self.P.T, self.T) / self.tau
        exp_logits = np.exp(self.logits - np.max(self.logits, axis=1, keepdims=True))
        self.softmax = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        
        target_probs = np.diag(self.softmax)
        return -np.mean(np.log(target_probs + 1e-8))

    def backward(self, S_y_hat):
        d_logits = self.softmax.copy()
        np.fill_diagonal(d_logits, np.diag(d_logits) - 1)
        d_logits /= self.batch_size
        d_P = np.dot(self.T, d_logits.T) / self.tau
        
        dot_dP_P = np.sum(d_P * self.P, axis=0, keepdims=True)
        return (d_P - self.P * dot_dP_P) / self.norm_P

# ==========================================
# 3. THE VL-JEPA MODEL (ViT + Text)
# ==========================================
class BatchedViTJEPA:
    def __init__(self, vit_encoder, text_encoder, vit_out_dim=256, shared_dim=64, lr=0.01):
        self.lr = lr
        self.vit = vit_encoder
        self.text_encoder = text_encoder
        self.criterion = InfoNCELoss(temperature=0.1)

        # Predictor Weights (maps ViT output down to Text Embedding dimension)
        self.W_p = np.random.randn(shared_dim, vit_out_dim) * np.sqrt(2. / vit_out_dim)
        self.b_p = np.zeros((shared_dim, 1))

    def forward(self, img_batch, text_tokens_batch):
        # 1. ViT Encoder (Frozen)
        tokens = self.vit.forward(img_batch) # Shape: (B, N, 256)
        self.S_v = tokens.mean(axis=1).T     # Pool to (256, B)

        # 2. Text Encoder (Frozen)
        S_y_list = [self.text_encoder.encode(tokens) for tokens in text_tokens_batch]
        self.S_y = np.hstack(S_y_list)       # Shape: (64, B)

        # 3. Predictor (Trainable)
        self.S_y_hat = np.dot(self.W_p, self.S_v) + self.b_p

        # 4. Loss
        return self.criterion.forward(self.S_y_hat, self.S_y)

    def backward(self):
        d_S_y_hat = self.criterion.backward(self.S_y_hat)
        d_W_p = np.dot(d_S_y_hat, self.S_v.T)
        d_b_p = np.sum(d_S_y_hat, axis=1, keepdims=True)
        
        # NOTE: We do NOT backpropagate into S_v. The ViT remains frozen!
        self.W_p -= self.lr * d_W_p
        self.b_p -= self.lr * d_b_p

# ==========================================
# 4. EVALUATION / PREDICTION
# ==========================================
def predict(model, feat, num_classes, text_mapping):
    # Prepare the fake image for inference
    feat = np.array(feat).flatten()
    feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
    
    img = np.tile(feat, 224*224*3 // len(feat) + 1)[:224*224*3]
    img = img.reshape(1, 224, 224, 3) # Batch size of 1 for inference

    # Pass through ViT and Predictor
    tokens = model.vit.forward(img)
    S_v = tokens.mean(axis=1).T
    S_y_hat = model.W_p @ S_v + model.b_p

    distances = []
    for c in range(num_classes):
        S_y = model.text_encoder.encode(text_mapping[c])
        # Cosine Similarity Distance
        sim = np.dot(S_y_hat.T, S_y) / (np.linalg.norm(S_y_hat) * np.linalg.norm(S_y) + 1e-8)
        distances.append(-sim.item())

    return np.argmin(distances)

# ==========================================
# 5. TRAINING LOOP
# ==========================================
def run_training_and_plot(csv_path, epochs=50, batch_size=4, num_classes=3):
    print(f"Loading features from {csv_path}...")
    df = pd.read_csv(csv_path)
    feature_col = 'features' if 'features' in df.columns else 'hsv_features'
    df[feature_col] = df[feature_col].apply(ast.literal_eval)

    text_mapping = {
        0: ["a","person","walking"],
        1: ["a","person","boxing"],
        2: ["a","person","clapping"]   
    }

    # Initialize ViT, Text Encoder, and Model
    vit = NumPyViT(img_size=224, patch_size=16, embed_dim=256, depth=4, heads=4)
    text_encoder = SimpleTextEncoder(vocab=vocab, embed_dim=64)
    model = BatchedViTJEPA(vit_encoder=vit, text_encoder=text_encoder, lr=0.05)
    
    loss_history = []
    print(f"Starting ViT-JEPA Training on {len(df)} samples...")

    for epoch in range(epochs):
        df_shuffled = df.sample(frac=1).reset_index(drop=True)
        epoch_loss = 0
        num_batches = 0

        for start_idx in range(0, len(df), batch_size):
            end_idx = min(start_idx + batch_size, len(df))
            batch_df = df_shuffled.iloc[start_idx:end_idx]
            
            if len(batch_df) < 2: continue # InfoNCE needs >= 2 samples

            img_batch = []
            text_tokens_batch = []
            
            # Format raw features into fake 4D image batch
            for _, row in batch_df.iterrows():
                feat = np.array(row[feature_col])
                feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
                
                img = np.tile(feat, 224*224*3 // len(feat) + 1)[:224*224*3]
                img = img.reshape(224, 224, 3) 
                
                img_batch.append(img)
                text_tokens_batch.append(text_mapping[int(row['target_class'])])

            img_batch = np.array(img_batch) # Final Shape: (Batch, 224, 224, 3)

            # Train Step
            loss = model.forward(img_batch, text_tokens_batch)
            model.backward()
            
            epoch_loss += loss
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        loss_history.append(avg_loss)
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

    # Evaluation
    correct = 0
    for _, row in df.iterrows():
        pred = predict(model, row[feature_col], num_classes, text_mapping)
        if pred == int(row['target_class']):
            correct += 1
            
    print(f"\nFinal Accuracy: {(correct/len(df))*100:.2f}%")

    # Plot
    plt.figure(figsize=(8,4))
    plt.plot(loss_history, color='purple')
    plt.title("VL-JEPA (ViT Encoder) Latent Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.show()

    return model

# ==========================================
# 6. EXECUTION
# ==========================================
if __name__ == "__main__":
    csv_file = r"C:\Users\subhe\OneDrive\Desktop\VL-JEPA\codes\VL-JEPA-base-CCTV\kth_spatial_motion3.csv"
    trained_model = run_training_and_plot(csv_file, epochs=50, batch_size=4)

In [ ]:
##Ayushman predictor architecture.

In [ ]:

import numpy as np

#RMSNorm
class NumPyRMSNorm:
    def __init__(self, embed_dim, learning_rate=0.001, eps=1e-6):
        self.eps = eps # A tiny number to prevent division by zero
        self.lr = learning_rate
        
        # Learnable scaling parameter (Gamma). Starts at 1.0
        # The network learns if it needs to scale specific dimensions up or down
        self.gamma = np.ones((embed_dim,))
        
        # Cache for backpropagation
        self.cache = {}

    def forward(self, x):
        """
        x: The combined sequence tokens H_0. Shape: (Sequence_Length, Embed_Dim)
        """
        # 1. Calculate the Root Mean Square of the sequence features
        # We square the elements, take the mean across the feature dimension, and add epsilon
        rms = np.sqrt(np.mean(x**2, axis=-1, keepdims=True) + self.eps)
        
        # 2. Normalize the input by dividing by the RMS
        x_norm = x / rms
        
        # 3. Scale by the learnable parameter gamma
        out = self.gamma * x_norm
        
        # Save variables for the calculus in the backward pass
        self.cache = {'x': x, 'x_norm': x_norm, 'rms': rms}
        
        return out

    def backward(self, d_out):
        """
        d_out: The gradient flowing backwards from the Attention layer
        """
        x = self.cache['x']
        x_norm = self.cache['x_norm']
        rms = self.cache['rms']
        N = x.shape[-1] # Number of features (embed_dim)
        
        # 1. Gradient with respect to the learnable parameter gamma
        # We sum across the sequence length axis because gamma applies to all tokens
        d_gamma = np.sum(d_out * x_norm, axis=0)
        
        # 2. Gradient with respect to the normalized input
        d_x_norm = d_out * self.gamma
        
        # 3. The chain rule calculus for RMS normalization
        # This calculates how the loss changes with respect to the original input 'x'
        d_rms = np.sum(d_x_norm * x * (-1.0 / (rms**2)), axis=-1, keepdims=True)
        d_x = (d_x_norm / rms) + (d_rms * x / (N * rms))
        
        # Update the learnable parameter
        self.gamma -= self.lr * d_gamma
        
        return d_x

#Multi Head Attention Layer

def stable_softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

class NumPyMultiHeadAttention:
    def __init__(self, embed_dim, num_heads, learning_rate=0.001):
        # Validation: embed_dim must be perfectly divisible by num_heads
        assert embed_dim % num_heads == 0, "Embedding dimension must be divisible by number of heads!"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads # e.g., 64 // 8 = 8 dimensions per head
        self.lr = learning_rate
        
        # The weight matrices remain the exact same size as single-head!
        self.W_q = np.random.randn(embed_dim, embed_dim) * np.sqrt(2. / embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) * np.sqrt(2. / embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) * np.sqrt(2. / embed_dim)
        self.W_o = np.random.randn(embed_dim, embed_dim) * np.sqrt(2. / embed_dim)
        
        self.cache = {}

    def forward(self, x):
        """
        x: The normalized concatenated sequence H_0 from the RMSNorm layer. 
        Shape: (Sequence_Length, Embed_Dim)
        """
        seq_len = x.shape[0]
        
        # 1. Linear Projections (Shape: Seq_Len x Embed_Dim)
        Q_full = np.dot(x, self.W_q)
        K_full = np.dot(x, self.W_k)
        V_full = np.dot(x, self.W_v)
        
        # 2. The Multi-Head Split & Transpose
        # We reshape from (Seq_Len, 64) -> (Seq_Len, 8 heads, 8 dims)
        # Then we transpose to (8 heads, Seq_Len, 8 dims) so the heads are independent batches
        Q = Q_full.reshape(seq_len, self.num_heads, self.head_dim).transpose(1, 0, 2)
        K = K_full.reshape(seq_len, self.num_heads, self.head_dim).transpose(1, 0, 2)
        V = V_full.reshape(seq_len, self.num_heads, self.head_dim).transpose(1, 0, 2)
        
        # 3. Scaled Dot-Product Attention across all heads simultaneously
        # K.transpose(0, 2, 1) flips the sequence and head_dim for the dot product
        # Resulting scores shape: (8 heads, Seq_Len, Seq_Len)
        scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(self.head_dim)
        
        # Apply Softmax across the last dimension (the key sequence length)
        attention_weights = stable_softmax(scores, axis=-1)
        
        # 4. Multiply weights by Values
        # Shape: (8 heads, Seq_Len, 8 dims)
        context_output = np.matmul(attention_weights, V)
        
        # 5. The Concatenation step
        # Transpose back to (Seq_Len, 8 heads, 8 dims), then flatten the last two dims back to 64
        context_output_flat = context_output.transpose(1, 0, 2).reshape(seq_len, self.embed_dim)
        
        # 6. Final Output Projection
        final_output = np.dot(context_output_flat, self.W_o)
        
        # Cache for the brutal backward pass
        self.cache = {
            'x': x, 'Q': Q, 'K': K, 'V': V, 
            'attention_weights': attention_weights, 
            'context_output_flat': context_output_flat
        }
        
        return final_output

    def backward(self, d_out):
        """
        Manual backpropagation through Multi-Head Attention tensors.
        d_out shape: (Sequence_Length, Embed_Dim)
        """
        x = self.cache['x']
        Q, K, V = self.cache['Q'], self.cache['K'], self.cache['V']
        attention_weights = self.cache['attention_weights']
        context_output_flat = self.cache['context_output_flat']
        seq_len = x.shape[0]

        # 1. Gradients for Output Projection
        d_W_o = np.dot(context_output_flat.T, d_out)
        d_context_flat = np.dot(d_out, self.W_o.T)

        # 2. Reshape gradient back into heads
        d_context = d_context_flat.reshape(seq_len, self.num_heads, self.head_dim).transpose(1, 0, 2)

        # 3. Gradients for V
        d_V = np.matmul(attention_weights.transpose(0, 2, 1), d_context)
        
        # 4. Gradients for Softmax and Scores
        d_attention_weights = np.matmul(d_context, V.transpose(0, 2, 1))
        
        # Softmax derivative trick inside the batched tensor
        d_scores = attention_weights * (d_attention_weights - np.sum(d_attention_weights * attention_weights, axis=-1, keepdims=True))
        d_scores = d_scores / np.sqrt(self.head_dim)

        # 5. Gradients for Q and K
        d_Q = np.matmul(d_scores, K)
        d_K = np.matmul(d_scores.transpose(0, 2, 1), Q)

        # 6. Re-flatten the split gradients back to original embed_dim
        d_Q_flat = d_Q.transpose(1, 0, 2).reshape(seq_len, self.embed_dim)
        d_K_flat = d_K.transpose(1, 0, 2).reshape(seq_len, self.embed_dim)
        d_V_flat = d_V.transpose(1, 0, 2).reshape(seq_len, self.embed_dim)

        # 7. Gradients for the input weight matrices
        d_W_q = np.dot(x.T, d_Q_flat)
        d_W_k = np.dot(x.T, d_K_flat)
        d_W_v = np.dot(x.T, d_V_flat)

        # 8. Gradient to pass down to the RMSNorm layer
        d_x = np.dot(d_Q_flat, self.W_q.T) + np.dot(d_K_flat, self.W_k.T) + np.dot(d_V_flat, self.W_v.T)

        # Apply weight updates
        self.W_q -= self.lr * d_W_q
        self.W_k -= self.lr * d_W_k
        self.W_v -= self.lr * d_W_v
        self.W_o -= self.lr * d_W_o

        return d_x

# --- SIMULATION TEST ---
# Test the layer with 8 heads!
embed_dimension = 64
num_attention_heads = 8

# Create the MHSA layer
mhsa_layer = NumPyMultiHeadAttention(embed_dim=embed_dimension, num_heads=num_attention_heads)

# Simulate the normalized sequence H_0 coming from RMSNorm (20 tokens total)
simulated_normalized_H0 = np.random.randn(20, embed_dimension)

# Forward pass
mhsa_output = mhsa_layer.forward(simulated_normalized_H0)

print(f"MHSA Output Shape: {mhsa_output.shape}") 
# Perfect success if it prints (20, 64)

import numpy as np

# --- 1. Stable SiLU Activation & Derivative ---
def sigmoid(x):
    # Clipped to prevent math overflow warnings in NumPy when exponents get too large
    x_clipped = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x_clipped))

def silu(x):
    # Swish (SiLU) is defined as x * sigmoid(x)
    return x * sigmoid(x)

def silu_derivative(x):
    # The calculus product-rule derivative of x * sigmoid(x)
    s = sigmoid(x)
    return s + x * s * (1.0 - s)

# --- 2. The SwiGLU FFN Class ---
class NumPySwiGLUFFN:
    def __init__(self, embed_dim, hidden_dim=None, learning_rate=0.001):
        self.embed_dim = embed_dim
        # In LLaMA, the hidden dimension usually expands to 4x the embed_dim 
        # to allow the network more "space" to reason non-linearly.
        self.hidden_dim = hidden_dim if hidden_dim is not None else embed_dim * 4
        self.lr = learning_rate
        
        # 1. The Gate Weight (Learns WHAT features to let through)
        self.W_gate = np.random.randn(embed_dim, self.hidden_dim) * np.sqrt(2. / embed_dim)
        
        # 2. The Up Weight (Learns the expanded feature representations)
        self.W_up = np.random.randn(embed_dim, self.hidden_dim) * np.sqrt(2. / embed_dim)
        
        # 3. The Down Weight (Compresses the reasoning back to original embed_dim)
        self.W_down = np.random.randn(self.hidden_dim, embed_dim) * np.sqrt(2. / self.hidden_dim)
        
        self.cache = {}

    def forward(self, x):
        """
        x: The sequence tokens coming out of the Attention layer.
        Shape: (Sequence_Length, Embed_Dim)
        """
        # Step 1: The two parallel linear projections
        gate_proj = np.dot(x, self.W_gate)  # Shape: (Seq_Len, Hidden_Dim)
        up_proj = np.dot(x, self.W_up)      # Shape: (Seq_Len, Hidden_Dim)
        
        # Step 2: Apply the Swish activation ONLY to the gate
        activated_gate = silu(gate_proj)
        
        # Step 3: The element-wise multiplication (The actual "Gating" mechanism)
        mid_representation = activated_gate * up_proj
        
        # Step 4: Compress back down to the original embedding dimension
        out = np.dot(mid_representation, self.W_down) # Shape: (Seq_Len, Embed_Dim)
        
        # Cache the variables for the backward pass
        self.cache = {
            'x': x,
            'gate_proj': gate_proj,
            'up_proj': up_proj,
            'activated_gate': activated_gate,
            'mid_representation': mid_representation
        }
        
        return out

    def backward(self, d_out):
        """
        Calculus for backpropagation through the parallel branches of SwiGLU.
        d_out shape: (Sequence_Length, Embed_Dim)
        """
        x = self.cache['x']
        gate_proj = self.cache['gate_proj']
        up_proj = self.cache['up_proj']
        activated_gate = self.cache['activated_gate']
        mid = self.cache['mid_representation']
        
        # 1. Gradients for the Down Projection matrix (W_down)
        d_W_down = np.dot(mid.T, d_out)
        d_mid = np.dot(d_out, self.W_down.T)  # Shape: (Seq_Len, Hidden_Dim)
        
        # 2. The gradient splits at the element-wise multiplication!
        # Product rule: derivative of (A * B) means d_A = d_out * B, and d_B = d_out * A
        d_activated_gate = d_mid * up_proj
        d_up_proj = d_mid * activated_gate
        
        # 3. Gradients for the Up Projection matrix (W_up)
        d_W_up = np.dot(x.T, d_up_proj)
        
        # 4. Gradients flowing through the SiLU activation function
        d_gate_proj = d_activated_gate * silu_derivative(gate_proj)
        
        # 5. Gradients for the Gate matrix (W_gate)
        d_W_gate = np.dot(x.T, d_gate_proj)
        
        # 6. Recombine the gradients to pass back to the Attention layer
        d_x = np.dot(d_gate_proj, self.W_gate.T) + np.dot(d_up_proj, self.W_up.T)
        
        # Apply the weight updates
        self.W_gate -= self.lr * d_W_gate
        self.W_up -= self.lr * d_W_up
        self.W_down -= self.lr * d_W_down
        
        return d_x
    
# --- SIMULATION TEST ---
# Let's test the Concatenation and RMSNorm together!
embed_dimension = 64

# Simulate 16 Visual tokens and 4 Query tokens
simulated_S_v = np.random.randn(16, embed_dimension)
simulated_S_q = np.random.randn(4, embed_dimension)

# 1. The Concatenation Step
H_0 = np.concatenate((simulated_S_v, simulated_S_q), axis=0)
print(f"Concatenated Sequence Shape: {H_0.shape}") # Should be (20, 64)

# 2. The RMSNorm Step
rmsnorm_layer = NumPyRMSNorm(embed_dim=embed_dimension)
normalized_H_0 = rmsnorm_layer.forward(H_0)

print(f"Normalized Sequence Shape: {normalized_H_0.shape}")

#NumPyTransformerBlock
class NumPyTransformerBlock:
    def __init__(self, embed_dim, num_heads, hidden_dim=None, learning_rate=0.001):
        # 1. First sub-layer: Attention
        self.norm1 = NumPyRMSNorm(embed_dim, learning_rate)
        self.attn = NumPyMultiHeadAttention(embed_dim, num_heads, learning_rate)
        
        # 2. Second sub-layer: Feed-Forward Network
        self.norm2 = NumPyRMSNorm(embed_dim, learning_rate)
        # (Assuming the NumPySwiGLUFFN class from our previous step is fully defined!)
        self.ffn = NumPySwiGLUFFN(embed_dim, hidden_dim, learning_rate)
        
        self.cache = {}

    def forward(self, x):
        """
        x: The sequence tokens. Shape: (Sequence_Length, Embed_Dim)
        """
        # --- SUB-LAYER 1: Multi-Head Attention ---
        # Pre-Norm
        norm1_out = self.norm1.forward(x)
        # Attention
        attn_out = self.attn.forward(norm1_out)
        # THE FIRST RESIDUAL CONNECTION
        x_mid = x + attn_out 
        
        # --- SUB-LAYER 2: SwiGLU FFN ---
        # Pre-Norm
        norm2_out = self.norm2.forward(x_mid)
        # FFN
        ffn_out = self.ffn.forward(norm2_out)
        # THE SECOND RESIDUAL CONNECTION
        x_out = x_mid + ffn_out 
        
        # Save cache for the backward pass
        self.cache = {
            'x': x,
            'norm1_out': norm1_out,
            'attn_out': attn_out,
            'x_mid': x_mid,
            'norm2_out': norm2_out,
            'ffn_out': ffn_out
        }
        
        return x_out

    def backward(self, d_out):
        """
        d_out: The gradient flowing backwards from the layer above.
        """
        # --- BACKPROP THROUGH SUB-LAYER 2 (FFN) ---
        # The gradient splits equally at the second residual connection!
        # d_out flows straight into the FFN branch AND straight down the skip branch
        d_ffn_out = d_out
        d_x_mid_from_skip2 = d_out 
        
        d_norm2_out = self.ffn.backward(d_ffn_out)
        d_x_mid_from_ffn = self.norm2.backward(d_norm2_out)
        
        # Combine the gradients that split at the second residual connection
        d_x_mid = d_x_mid_from_skip2 + d_x_mid_from_ffn
        
        # --- BACKPROP THROUGH SUB-LAYER 1 (Attention) ---
        # The gradient splits equally at the first residual connection!
        d_attn_out = d_x_mid
        d_x_from_skip1 = d_x_mid
        
        d_norm1_out = self.attn.backward(d_attn_out)
        d_x_from_attn = self.norm1.backward(d_norm1_out)
        
        # Combine the gradients that split at the first residual connection
        d_x = d_x_from_skip1 + d_x_from_attn
        
        return d_x

class NumPyVLJEPAPredictor:
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers=4, hidden_dim=None, seq_length_q=4, lr=0.001):
        self.embed_dim = embed_dim
        self.lr = lr
        
        # 1. The Text Embedder (Converts raw X_q into embedded S_q)
        # (Assuming NumPyTextEmbedder is defined from our previous step)
        self.embedder = NumPyTextEmbedder(vocab_size, embed_dim, max_seq_length=seq_length_q, learning_rate=lr)
        
        # 2. The Transformer Stack (The LLaMA-style engine)
        self.blocks = []
        for _ in range(num_layers):
            # (Assuming NumPyTransformerBlock is defined from our previous step)
            self.blocks.append(NumPyTransformerBlock(embed_dim, num_heads, hidden_dim, lr))
            
        self.cache = {}

    def forward(self, S_v, X_q_tokens):
        """
        S_v: The visible visual tokens from the X-Encoder. Shape: (Seq_Len_V, Embed_Dim)
        X_q_tokens: The tokenized textual query. Shape: (Seq_Len_Q,)
        """
        seq_len_v = S_v.shape[0]
        seq_len_q = len(X_q_tokens)
        
        # Phase 1: Embed the Textual Query
        S_q = self.embedder.forward(X_q_tokens)
        
        # Phase 2: Concatenate into a single continuous sequence
        # H_0 = [S_v ; S_q]
        H = np.concatenate((S_v, S_q), axis=0) 
        
        # Phase 3: Pass through the Transformer layers
        for block in self.blocks:
            H = block.forward(H)
            
        # Phase 4: Extract ONLY the transformed Query tokens
        # Because we appended S_q to the end of S_v, the query tokens are at the bottom of the matrix
        H_q_out = H[seq_len_v:, :] 
        
        # Phase 5: Average Pooling
        # Collapse the sequence of query tokens into a single predictive vector
        S_y_hat = np.mean(H_q_out, axis=0, keepdims=True) # Shape: (1, Embed_Dim)
        
        # Cache dimensions for the backward pass
        self.cache = {
            'seq_len_v': seq_len_v,
            'seq_len_q': seq_len_q
        }
        
        return S_y_hat

    def backward(self, d_S_y_hat):
        """
        d_S_y_hat: The gradient flowing back from your InfoNCE Loss. 
        Shape: (1, Embed_Dim)
        """
        seq_len_v = self.cache['seq_len_v']
        seq_len_q = self.cache['seq_len_q']
        
        # Phase 1: Reverse the Average Pooling 
        # Since the forward pass averaged 'n' tokens, the backward pass distributes 
        # the gradient equally divided by 'n' across all query tokens.
        d_H_q_out = np.repeat(d_S_y_hat / seq_len_q, seq_len_q, axis=0)
        
        # Phase 2: Reconstruct the full sequence gradient (d_H)
        # The loss was ONLY calculated on the query prediction. The visual tokens 
        # did not directly contribute to the final loss vector, so their initial gradient is zero.
        d_H_v_out = np.zeros((seq_len_v, self.embed_dim))
        d_H = np.concatenate((d_H_v_out, d_H_q_out), axis=0)
        
        # Phase 3: Backpropagate through the Transformer Stack in reverse order
        for block in reversed(self.blocks):
            d_H = block.backward(d_H)
            
        # Phase 4: Split the gradient back into Visual and Textual pathways
        # Even though the visual gradient started as zeros, the Attention mechanism 
        # mathematically mixed the data, meaning d_S_v will now contain rich gradients!
        d_S_v = d_H[:seq_len_v, :]
        d_S_q = d_H[seq_len_v:, :]
        
        # Phase 5: Backpropagate through the Embedder to update word embeddings
        self.embedder.backward(d_S_q)
        
        # Return d_S_v to the main training loop so it can update your X-Encoder!
        return d_S_v
    
# --- 1. The UPGRADED Transformer JEPA Class ---
class TransformerNumPyJEPA:
    def __init__(self, input_dim, target_classes, embed_dim=64, num_heads=4, num_layers=2, lr=0.002):
        self.lr = lr
        self.embed_dim = embed_dim
        
        # 1. Visual Encoder (Stays the same!)
        self.W_v = np.random.randn(embed_dim, input_dim) * np.sqrt(2. / input_dim)
        self.b_v = np.zeros((embed_dim, 1))
        
        # 2. Target Label Encoder (Stays the same!)
        self.W_y = np.random.randn(embed_dim, target_classes) * np.sqrt(2. / target_classes)
        self.b_y = np.zeros((embed_dim, 1))
        
        # 3. THE NEW PREDICTOR: LLaMA Transformer Stack!
        # vocab_size=1 because we only need 1 dummy token to act as the "query"
        self.predictor = NumPyVLJEPAPredictor(
            vocab_size=1, embed_dim=embed_dim, num_heads=num_heads, 
            num_layers=num_layers, seq_length_q=1, lr=lr
        )

    def relu(self, Z): return np.maximum(0, Z)
    def relu_deriv(self, Z): return Z > 0

    def forward(self, X_v, Y):
        # 1. Visual Encoder -> S_v (Shape: 64, 1)
        Z_v = np.dot(self.W_v, X_v) + self.b_v
        self.S_v = self.relu(Z_v)
        
        # TRANSFORMATION: Convert (64, 1) to a Sequence (1, 64) for the Transformer
        S_v_seq = self.S_v.T
        
        # 2. Target Encoder -> S_y (Shape: 64, 1)
        self.S_y = np.dot(self.W_y, Y) + self.b_y
        S_y_seq = self.S_y.T
        
        # 3. Transformer Predictor
        # We pass the visual sequence and ask it to fill in the dummy query token [0]
        dummy_query = np.array([0])
        self.S_y_hat_seq = self.predictor.forward(S_v_seq, dummy_query)
        
        # 4. JEPA Embedding Loss
        loss = 0.5 * np.sum((self.S_y_hat_seq - S_y_seq)**2)
        return loss, Z_v

    def backward(self, X_v, Y, Z_v):
        S_y_seq = self.S_y.T
        
        # 1. Gradient of the Loss (Shape: 1, 64)
        d_S_y_hat_seq = self.S_y_hat_seq - S_y_seq
        
        # 2. The Heavy Calculus: Backprop through the entire Transformer stack!
        d_S_v_seq = self.predictor.backward(d_S_y_hat_seq)
        
        # TRANSFORMATION: Convert Transformer sequence (1, 64) back to column vectors (64, 1)
        d_S_v = d_S_v_seq.T
        d_S_y = -d_S_y_hat_seq.T
        
        # 3. Backprop through Visual Encoder
        d_Z_v = d_S_v * self.relu_deriv(Z_v)
        d_W_v = np.dot(d_Z_v, X_v.T)
        d_b_v = d_Z_v
        
        # 4. Backprop through Target Encoder
        d_W_y = np.dot(d_S_y, Y.T)
        d_b_y = d_S_y
        
        # 5. Weight Updates
        self.W_v -= self.lr * d_W_v; self.b_v -= self.lr * d_b_v
        self.W_y -= self.lr * d_W_y; self.b_y -= self.lr * d_b_y

# --- 2. Inference Helper (Updated for Transformer) ---
def predict_transformer(model, X_v, num_classes):
    # Encode Visual
    Z_v = np.dot(model.W_v, X_v) + model.b_v
    S_v = model.relu(Z_v)
    
    # Predict with Transformer
    S_v_seq = S_v.T
    dummy_query = np.array([0])
    S_y_hat_seq = model.predictor.forward(S_v_seq, dummy_query)
    
    # Find the nearest neighbor class
    distances = []
    for i in range(num_classes):
        Y_t = np.zeros((num_classes, 1))
        Y_t[i, 0] = 1.0
        S_y_target = np.dot(model.W_y, Y_t) + model.b_y
        distances.append(np.linalg.norm(S_y_hat_seq - S_y_target.T))
    return np.argmin(distances)

# --- 3. Training Logic ---
def run_transformer_training_and_plot(csv_path, num_classes=3, epochs=50):
    print(f"Loading Motion Features from {csv_path}...")
    try:
        df = pd.read_csv(csv_path)
        feature_col = 'features' if 'features' in df.columns else 'hsv_features'
        df[feature_col] = df[feature_col].apply(ast.literal_eval)
    except Exception as e:
        print(f"Error: {e}")
        return

    # Initialize the new TRANSFORMER Model
    # We use 4 attention heads and 2 LLaMA layers. 
    model = TransformerNumPyJEPA(input_dim=256, target_classes=num_classes, embed_dim=64, num_heads=4, num_layers=2, lr=0.002)
    loss_history = []

    print(f"Starting LLaMA-Style Transformer Training on {len(df)} samples...")
    for epoch in range(epochs):
        df_shuffled = df.sample(frac=1).reset_index(drop=True)
        epoch_loss = 0

        for _, row in df_shuffled.iterrows():
            X_v = np.array(row[feature_col]).reshape(256, 1)
            Y = np.zeros((num_classes, 1))
            Y[int(row['target_class']), 0] = 1.0

            # Forward and Backward passes now flow through the Transformer!
            loss, Z_v = model.forward(X_v, Y)
            model.backward(X_v, Y, Z_v)
            epoch_loss += loss

        avg_loss = epoch_loss / len(df)
        loss_history.append(avg_loss)
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | Transformer JEPA Loss: {avg_loss:.6f}")

    # --- Evaluation Phase ---
    correct = 0
    for _, row in df.iterrows():
        X_v = np.array(row[feature_col]).reshape(256, 1)
        pred = predict_transformer(model, X_v, num_classes)
        if pred == int(row['target_class']):
            correct += 1
    
    print(f"\n--- Final Results ---")
    print(f"Transformer Accuracy: {(correct/len(df))*100:.2f}%")

    # Plotting
    plt.figure(figsize=(8, 4))
    plt.plot(loss_history, color='purple', linewidth=2)
    plt.title('NumPy LLaMA-JEPA: Training Loss (Attention Mechanism)')
    plt.xlabel('Epochs')
    plt.ylabel('L2 Latent Distance')
    plt.grid(True, alpha=0.5)
    plt.show()
    
    return model

# --- 4. Execution ---
csv_file = r"C:\Users\Ayushman\VL-JEPA-base-CCTV\kth_motion_features2.csv"
trained_transformer_model = run_transformer_training_and_plot(csv_file, num_classes=3, epochs=50)

# ViT + Text encoder

In [ ]:
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt

# ==========================================
# 1. VISION TRANSFORMER (X-Encoder)
# ==========================================
class PatchEmbedding:
    def __init__(self, img_size=224, patch_size=16, in_ch=3, embed_dim=256):
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.num_patches = (img_size // patch_size) ** 2
        patch_dim = patch_size * patch_size * in_ch
        self.W = np.random.randn(patch_dim, embed_dim) / np.sqrt(patch_dim)

    def forward(self, x):
        B, H, W, C = x.shape
        p = self.patch_size
        patches = []
        for i in range(0, H, p):
            for j in range(0, W, p):
                patch = x[:, i:i+p, j:j+p, :].reshape(B, -1)
                patches.append(patch)
        patches = np.stack(patches, axis=1)   # (B, N, patch_dim)
        return patches @ self.W               # (B, N, embed_dim)

class LayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.gamma = np.ones((1, 1, dim))
        self.beta = np.zeros((1, 1, dim))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

class MLP:
    def __init__(self, dim, hidden_dim):
        self.W1 = np.random.randn(dim, hidden_dim) / np.sqrt(dim)
        self.W2 = np.random.randn(hidden_dim, dim) / np.sqrt(hidden_dim)

    def gelu(self, x):
        return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * np.power(x, 3))))

    def forward(self, x):
        return self.gelu(x @ self.W1) @ self.W2

class MultiHeadSelfAttention:
    def __init__(self, dim, heads=8):
        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.Wq = np.random.randn(dim, dim) / np.sqrt(dim)
        self.Wk = np.random.randn(dim, dim) / np.sqrt(dim)
        self.Wv = np.random.randn(dim, dim) / np.sqrt(dim)
        self.Wo = np.random.randn(dim, dim) / np.sqrt(dim)

    def forward(self, x):
        B, N, D = x.shape
        Q = x @ self.Wq
        K = x @ self.Wk
        V = x @ self.Wv

        Q = Q.reshape(B, N, self.heads, self.head_dim).transpose(0,2,1,3)
        K = K.reshape(B, N, self.heads, self.head_dim).transpose(0,2,1,3)
        V = V.reshape(B, N, self.heads, self.head_dim).transpose(0,2,1,3)

        scores = Q @ K.transpose(0,1,3,2) / np.sqrt(self.head_dim)
        
        # Stability adjustment for Softmax
        scores = scores - np.max(scores, axis=-1, keepdims=True)
        attn = np.exp(scores)
        attn = attn / attn.sum(axis=-1, keepdims=True)

        out = attn @ V
        out = out.transpose(0,2,1,3).reshape(B, N, D)
        return out @ self.Wo

class TransformerBlock:
    def __init__(self, dim, heads):
        self.norm1 = LayerNorm(dim)
        self.norm2 = LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, heads)
        self.mlp = MLP(dim, dim*4)

    def forward(self, x):
        x = x + self.attn.forward(self.norm1.forward(x))
        x = x + self.mlp.forward(self.norm2.forward(x))
        return x

class NumPyViT:
    def __init__(self, img_size=224, patch_size=16, embed_dim=256, depth=4, heads=4):
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        self.pos_embed = np.random.randn(1, self.patch_embed.num_patches, embed_dim)
        self.blocks = [TransformerBlock(embed_dim, heads) for _ in range(depth)]
        self.norm = LayerNorm(embed_dim)

    def forward(self, x):
        x = self.patch_embed.forward(x)
        x = x + self.pos_embed
        for blk in self.blocks:
            x = blk.forward(x)
        return self.norm.forward(x)

# ==========================================
# 2. TEXT ENCODER & INFONCE LOSS
# ==========================================
vocab = ["a", "person", "walking", "boxing", "clapping"]

class SimpleTextEncoder:
    def __init__(self, vocab, embed_dim=64): 
        np.random.seed(42)
        self.embed_dim = embed_dim
        self.word_to_idx = {w:i for i,w in enumerate(vocab)}
        self.embeddings = np.random.randn(len(vocab), embed_dim) * 1.0 

    def encode(self, words):
        idxs = [self.word_to_idx[w] for w in words]
        sentence = np.mean(self.embeddings[idxs], axis=0)
        return sentence.reshape(-1, 1)

class InfoNCELoss:
    def __init__(self, temperature=0.1):
        self.tau = temperature

    def forward(self, S_y_hat, S_y):
        self.batch_size = S_y_hat.shape[1]
        self.norm_P = np.linalg.norm(S_y_hat, axis=0, keepdims=True) + 1e-8
        self.norm_T = np.linalg.norm(S_y, axis=0, keepdims=True) + 1e-8
        
        self.P = S_y_hat / self.norm_P
        self.T = S_y / self.norm_T
        
        self.logits = np.dot(self.P.T, self.T) / self.tau
        exp_logits = np.exp(self.logits - np.max(self.logits, axis=1, keepdims=True))
        self.softmax = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        
        target_probs = np.diag(self.softmax)
        return -np.mean(np.log(target_probs + 1e-8))

    def backward(self, S_y_hat):
        d_logits = self.softmax.copy()
        np.fill_diagonal(d_logits, np.diag(d_logits) - 1)
        d_logits /= self.batch_size
        d_P = np.dot(self.T, d_logits.T) / self.tau
        
        dot_dP_P = np.sum(d_P * self.P, axis=0, keepdims=True)
        return (d_P - self.P * dot_dP_P) / self.norm_P

# ==========================================
# 3. THE VL-JEPA MODEL (ViT + Text)
# ==========================================
class BatchedViTJEPA:
    def __init__(self, vit_encoder, text_encoder, vit_out_dim=256, shared_dim=64, lr=0.01):
        self.lr = lr
        self.vit = vit_encoder
        self.text_encoder = text_encoder
        self.criterion = InfoNCELoss(temperature=0.1)

        # Predictor Weights (maps ViT output down to Text Embedding dimension)
        self.W_p = np.random.randn(shared_dim, vit_out_dim) * np.sqrt(2. / vit_out_dim)
        self.b_p = np.zeros((shared_dim, 1))

    def forward(self, img_batch, text_tokens_batch):
        # 1. ViT Encoder (Frozen)
        tokens = self.vit.forward(img_batch) # Shape: (B, N, 256)
        self.S_v = tokens.mean(axis=1).T     # Pool to (256, B)

        # 2. Text Encoder (Frozen)
        S_y_list = [self.text_encoder.encode(tokens) for tokens in text_tokens_batch]
        self.S_y = np.hstack(S_y_list)       # Shape: (64, B)

        # 3. Predictor (Trainable)
        self.S_y_hat = np.dot(self.W_p, self.S_v) + self.b_p

        # 4. Loss
        return self.criterion.forward(self.S_y_hat, self.S_y)

    def backward(self):
        d_S_y_hat = self.criterion.backward(self.S_y_hat)
        d_W_p = np.dot(d_S_y_hat, self.S_v.T)
        d_b_p = np.sum(d_S_y_hat, axis=1, keepdims=True)
        
        # NOTE: We do NOT backpropagate into S_v. The ViT remains frozen!
        self.W_p -= self.lr * d_W_p
        self.b_p -= self.lr * d_b_p

# ==========================================
# 4. EVALUATION / PREDICTION
# ==========================================
def predict(model, feat, num_classes, text_mapping):
    # Prepare the fake image for inference
    feat = np.array(feat).flatten()
    feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
    
    img = np.tile(feat, 224*224*3 // len(feat) + 1)[:224*224*3]
    img = img.reshape(1, 224, 224, 3) # Batch size of 1 for inference

    # Pass through ViT and Predictor
    tokens = model.vit.forward(img)
    S_v = tokens.mean(axis=1).T
    S_y_hat = model.W_p @ S_v + model.b_p

    distances = []
    for c in range(num_classes):
        S_y = model.text_encoder.encode(text_mapping[c])
        # Cosine Similarity Distance
        sim = np.dot(S_y_hat.T, S_y) / (np.linalg.norm(S_y_hat) * np.linalg.norm(S_y) + 1e-8)
        distances.append(-sim.item())

    return np.argmin(distances)

# ==========================================
# 5. TRAINING LOOP
# ==========================================
def run_training_and_plot(csv_path, epochs=50, batch_size=4, num_classes=3):
    print(f"Loading features from {csv_path}...")
    df = pd.read_csv(csv_path)
    feature_col = 'features' if 'features' in df.columns else 'hsv_features'
    df[feature_col] = df[feature_col].apply(ast.literal_eval)

    text_mapping = {
        0: ["a","person","walking"],
        1: ["a","person","boxing"],
        2: ["a","person","clapping"]   
    }

    # Initialize ViT, Text Encoder, and Model
    vit = NumPyViT(img_size=224, patch_size=16, embed_dim=256, depth=4, heads=4)
    text_encoder = SimpleTextEncoder(vocab=vocab, embed_dim=64)
    model = BatchedViTJEPA(vit_encoder=vit, text_encoder=text_encoder, lr=0.05)
    
    loss_history = []
    print(f"Starting ViT-JEPA Training on {len(df)} samples...")

    for epoch in range(epochs):
        df_shuffled = df.sample(frac=1).reset_index(drop=True)
        epoch_loss = 0
        num_batches = 0

        for start_idx in range(0, len(df), batch_size):
            end_idx = min(start_idx + batch_size, len(df))
            batch_df = df_shuffled.iloc[start_idx:end_idx]
            
            if len(batch_df) < 2: continue # InfoNCE needs >= 2 samples

            img_batch = []
            text_tokens_batch = []
            
            # Format raw features into fake 4D image batch
            for _, row in batch_df.iterrows():
                feat = np.array(row[feature_col])
                feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
                
                img = np.tile(feat, 224*224*3 // len(feat) + 1)[:224*224*3]
                img = img.reshape(224, 224, 3) 
                
                img_batch.append(img)
                text_tokens_batch.append(text_mapping[int(row['target_class'])])

            img_batch = np.array(img_batch) # Final Shape: (Batch, 224, 224, 3)

            # Train Step
            loss = model.forward(img_batch, text_tokens_batch)
            model.backward()
            
            epoch_loss += loss
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        loss_history.append(avg_loss)
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

    # Evaluation
    correct = 0
    for _, row in df.iterrows():
        pred = predict(model, row[feature_col], num_classes, text_mapping)
        if pred == int(row['target_class']):
            correct += 1
            
    print(f"\nFinal Accuracy: {(correct/len(df))*100:.2f}%")

    # Plot
    plt.figure(figsize=(8,4))
    plt.plot(loss_history, color='purple')
    plt.title("VL-JEPA (ViT Encoder) Latent Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.show()

    return model

# ==========================================
# 6. EXECUTION
# ==========================================
if __name__ == "__main__":
    csv_file = r"C:\Users\Hp\OneDrive\VL-JEPA-base-CCTV\kth_spatial_motion3.csv"
    trained_model = run_training_and_plot(csv_file, epochs=50, batch_size=4)